# Diabetes and Community Health

County-level descriptive analysis using the CDC PLACES 2025 release. The measures are modeled estimates, not individual-level records. This notebook focuses on association and does not estimate causal effects.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path('data/cdc_places_county_selected_2025.csv')
raw = pd.read_csv(DATA, dtype={'locationid': 'string'})
raw = raw[(raw.stateabbr != 'US') & raw.locationid.str.len().isin([4, 5])].copy()
raw['locationid'] = raw.locationid.str.zfill(5)
wide = (raw.pivot(index=['locationid', 'stateabbr', 'statedesc', 'locationname'],
                  columns='measureid', values='data_value')
          .reset_index())
wide.columns.name = None
wide = wide.rename(columns={'locationid':'county_fips', 'stateabbr':'state_abbr',
                            'statedesc':'state', 'locationname':'county',
                            'DIABETES':'diabetes_pct', 'OBESITY':'obesity_pct',
                            'LPA':'inactive_pct', 'ACCESS2':'uninsured_pct',
                            'FOODINSECU':'food_insecurity_pct'})
wide.shape

## Coverage and distribution


In [ ]:
base = wide.dropna(subset=['diabetes_pct', 'obesity_pct', 'inactive_pct', 'uninsured_pct']).copy()
food = wide.dropna(subset=['diabetes_pct', 'food_insecurity_pct']).copy()
summary = base.diabetes_pct.agg(['count', 'min', 'median', 'mean', 'max'])
summary

## Associations

We report both the pooled county correlation and a within-state correlation. The latter first subtracts each state’s mean from each county measure; it describes whether counties with higher values than their state average also tend to have higher diabetes prevalence. Neither is causal.


In [ ]:
measures = ['obesity_pct', 'inactive_pct', 'uninsured_pct']
pooled = base[['diabetes_pct'] + measures].corr()['diabetes_pct'].drop('diabetes_pct')
state_means = base.groupby('state_abbr')[['diabetes_pct'] + measures].transform('mean')
within = base[['diabetes_pct'] + measures] - state_means
within_corr = within.corr()['diabetes_pct'].drop('diabetes_pct')
food_r = food[['diabetes_pct', 'food_insecurity_pct']].corr().iloc[0, 1]
pd.DataFrame({'pooled_pearson_r': pooled, 'within_state_pearson_r': within_corr})

## Geographic summary

The following state ranking is an unweighted average of county estimates. Because counties differ greatly in population, it is not the state’s population-weighted prevalence.


In [ ]:
state_summary = (base.groupby(['state_abbr','state'], as_index=False)
  .agg(mean_county_diabetes_pct=('diabetes_pct','mean'), counties=('county_fips','count'))
  .sort_values('mean_county_diabetes_pct', ascending=False))
state_summary.head(10)

## Caveats

PLACES estimates are model-based small-area estimates based on BRFSS and supporting population data. All measures here are county-level and cross-sectional. Ecological correlations do not imply individual risk or causation. Food insecurity is available for fewer counties than the four core measures. See `findings.md` for the current summary and `README.md` for data provenance.
